# TP Jour 1 — Choix des Datasets Kaggle (Mono ou Multi-Sources), HDFS & Couche Bronze

Bienvenue dans ce fil rouge **Data Lakehouse** ! Vous êtes libres de choisir votre thématique métier et de combiner **un dataset multi-tables ou plusieurs datasets publics complémentaires sur Kaggle** afin de les croiser par jointure.

### Objectifs de la séance
1. **Cadrer votre sujet métier** et sélectionner **un ou plusieurs datasets Kaggle** (~50 à 500 Mo au total) partageant au moins un axe de jointure (date/heure, zone géographique, code ISO/catégorie, identifiant métier).
2. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) configuré avec `dfs.blocksize = 16 Mo` et `dfs.replication = 2`.
3. **Ingérer vos fichiers bruts multi-sources sur HDFS** (`/lakehouse/bronze/raw/<source>/...`) et observer physiquement leur découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : couper `datanode2` et observer la détection de panne + l'auto-réparation vers le 3e DataNode en ~50 secondes.
5. **Construire votre couche Bronze** (`/lakehouse/bronze/`) au format **Delta Lake** avec traçabilité de la source (`_source_dataset`, `_source_file`, `_ingested_at`) et inspecter vos jobs sur **Spark Web UI** (`http://localhost:4040`).

---
### Fiche de cadrage de votre projet *(À compléter par le groupe)*
- **Titre du sujet :** `...`
- **Dataset(s) Kaggle retenu(s) (`auteur/nom-du-dataset`) :**
  1. Source principale : `...`
  2. Source(s) complémentaire(s) pour jointure : `...`
- **Clé(s) de jointure envisagée(s) entre sources / tables :** `...` *(ex. date + ville, code aéroport, code postal, catégorie...)*
- **Problématique métier principale :** `...`
- **Variables clés retenues (temporelles, quantitatives, catégorielles) :** `...`

## 1. Téléchargement de vos Datasets Kaggle dans `/workspace/data_raw/<source>/`

Renseignez dans la liste `KAGGLE_DATASETS` un ou plusieurs identifiants Kaggle (ex. `["auteur1/dataset-principal", "auteur2/meteo-ou-referentiel"]`), ou déposez directement vos fichiers/sous-dossiers dans `data_raw/`.
- Chaque dataset est isolé dans son propre sous-dossier `data_raw/<nom-dataset>/` pour éviter toute collision de noms (`train.csv`, `metadata.csv`, etc.) entre plusieurs sources.
- **Critère important :** Vérifiez qu'au moins un fichier dépasse **20 à 32 Mo** afin d'observer sa division en plusieurs blocs HDFS de **16 Mo** !

In [29]:
import re
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

# TODO : Renseignez un ou plusieurs datasets Kaggle à combiner (format "auteur/nom-du-dataset")
KAGGLE_DATASETS: list[str] = [
    # Exemple source 1 (faits)       : "auteur1/dataset-principal",
    # Exemple source 2 (jointure)    : "auteur2/dataset-meteo-ou-geo",
]

for slug in KAGGLE_DATASETS:
    slug = slug.strip()
    if not slug:
        continue
    subdir = re.sub(r"[^a-zA-Z0-9._-]+", "_", slug.split("/")[-1]).strip("_") or "dataset"
    target_dir = DATA_RAW / subdir
    existing = [f for f in target_dir.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS] if target_dir.exists() else []
    if not existing:
        import kagglehub
        print(f"Téléchargement de {slug} -> {target_dir}/ ...")
        cache_dir = Path(kagglehub.dataset_download(slug))
        target_dir.mkdir(parents=True, exist_ok=True)
        for src_file in sorted(cache_dir.rglob("*")):
            if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
                dest_file = target_dir / src_file.relative_to(cache_dir)
                dest_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_file, dest_file)

local_files = sorted([f for f in DATA_RAW.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

assert len(local_files) > 0, (
    "Aucun fichier détecté dans /workspace/data_raw/. "
    "Renseignez `KAGGLE_DATASETS` ci-dessus ou utilisez `task data:download DATASET='auteur1/ds1,auteur2/ds2'`."
)

print(f"Inventaire local multi-sources dans {DATA_RAW} ({len(local_files)} fichier(s)) :")
for f in local_files:
    rel = f.relative_to(DATA_RAW)
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {str(rel):<55} {size_mb:>7.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

Inventaire local multi-sources dans /workspace/data_raw (4 fichier(s)) :
 - airlines.csv                                               0.00 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - airports.csv                                               0.02 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - cancellation_codes.csv                                     0.00 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - flights.csv                                              564.96 Mo  (~36 bloc(s) HDFS de 16 Mo)


## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [30]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# TODO : Initialiser la SparkSession
spark = (
    SparkSession.builder
    .appName("HDFS Ingestion Bronze")
    .getOrCreate()
)

assert spark is not None, "TODO : Initialisez `spark` avec SparkSession.builder..."

# TODO : Lire les 3 paramètres HDFS depuis `spark.conf.get(...)`
default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = spark.conf.get("spark.hadoop.dfs.blocksize")
replication = spark.conf.get("spark.hadoop.dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)

Spark version       : 4.2.0
Default FileSystem  : hdfs://namenode:9000
HDFS Block Size (o) : 16777216
HDFS Replication    : 2


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [31]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    rel_posix = file_path.relative_to(DATA_RAW).as_posix()
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{rel_posix}")
    fs.mkdirs(dst.getParent())
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

Téléversé vers HDFS : /lakehouse/bronze/raw/airlines.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/airports.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/cancellation_codes.csv


Téléversé vers HDFS : /lakehouse/bronze/raw/flights.csv


In [32]:
# Inspection récursive du découpage en blocs de 16 Mo sur l'ensemble des sources ingérées
remote_iter = fs.listFiles(raw_hdfs_dir, True)
statuses = []
while remote_iter.hasNext():
    statuses.append(remote_iter.next())
statuses.sort(key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    rel_hdfs = path.toString().split("/lakehouse/bronze/raw/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {rel_hdfs} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

Fichier : flights.csv (564.96 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #2 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #3 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #4 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #5 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #6 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #7 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #8 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #9 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #10 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #11 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #12 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #13 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #14 (16.00 Mo) -> DataNodes : ['datanode3', 'data

### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.

Le fichier le plus volumineux est flights.csv, avec une taille de 564,96 Mio. La taille maximale d'un bloc HDFS est configurée à 16 Mio (16 777 216 octets).
En divisant 564,96 par 16, nous obtenons 35,31, soit 36 blocs après arrondi à l'entier supérieur.
Les résultats confirment ce calcul : HDFS a créé 36 blocs, dont 35 blocs de 16 Mio et un dernier bloc de 4,96 Mio.
Le découpage observé est donc cohérent avec la configuration HDFS.

2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?

Non. La taille de 16 Mo représente la taille maximale configurée pour un bloc HDFS. Un fichier de 3 Mo occupe environ 3 Mo par copie, et non 16 Mo. Avec un facteur de réplication de 2, il occupe donc environ 6 Mo au total.

3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?

Le facteur de réplication de 2 permet de conserver deux copies de chaque bloc sur des DataNodes différents ce qui permet d'avoir une copie disponible si un DataNode tombe en panne. Le NameNode peut ensuite demander la création d'une nouvelle copie sur un DataNode encore actif pour rétablir le facteur de réplication. 


---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

Le fichier flights.csv, d'une taille de 564,96 Mio, est découpé en 36 blocs HDFS de 16 Mio maximum, avec un facteur de réplication de 2.
Lors de sa lecture avec PySpark, nous avons obtenu 5 819 080 lignes et 16 partitions Spark.
Le nombre de partitions Spark est donc différent du nombre de blocs HDFS. Cette différence s'explique par leurs rôles distincts : HDFS découpe les fichiers en blocs pour les stocker et les répliquer sur les DataNodes, tandis que Spark crée des partitions pour organiser le traitement des données en parallèle. Une partition Spark peut regrouper les données de plusieurs blocs HDFS. Il n'existe donc pas de correspondance systématique entre les blocs HDFS et les partitions Spark.
La lecture effectuée pendant l'arrêt du datanode2 montre la récupération des 5 819 080 lignes, ce qui signifie que les données restent accessibles malgfé la panne d'un DataNode, grâce à la réplicattion des blocs HDFS.


*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [33]:
print("Fichier cible du test de résilience :", largest_hdfs_path)

# TODO : Lire `largest_hdfs_path` avec PySpark (selon son format : spark.read.csv / parquet / json)
df_check = spark.read.csv(largest_hdfs_path)

assert df_check is not None, "TODO : Lisez `largest_hdfs_path` dans `df_check`."

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")

Fichier cible du test de résilience : hdfs://namenode:9000/lakehouse/bronze/raw/flights.csv
Nombre total de lignes lues       : 5,819,080
Nombre de partitions Spark créées : 16


---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts (mono ou multi-datasets) en tables Delta Lake Bronze
Dans une architecture **Médaillon multi-sources**, la couche **Bronze** conserve chaque table brute fidèle à sa source tout en l'enrichissant de métadonnées de lignage (*lineage*) indispensables avant les jointures du Jour 2 :
- `_ingested_at` : `F.current_timestamp()`
- `_source_dataset` : `F.lit(dataset_name)` *(identifie le dataset d'origine)*
- `_source_file` : `F.col("_metadata.file_path")`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de vos datasets dans `/lakehouse/bronze/<nom_table>`.

In [36]:
for f in local_files:
    rel = f.relative_to(DATA_RAW)
    dataset_name = rel.parts[0] if len(rel.parts) > 1 else "default"
    table_name = f.stem.lower().replace(" ", "_")
    raw_path = f"/lakehouse/bronze/raw/{rel.as_posix()}"
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"

    df_raw = (
        spark.read.parquet(raw_path)
        if f.suffix.lower() == ".parquet"
        else spark.read.option("header", "true").option("inferSchema", "true").option("multiLine", "true").option("escape", '"').csv(raw_path)
    )
    df_raw = (
        df_raw.toDF(*[c.strip().lower().replace(" ", "_") for c in df_raw.columns])
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_dataset", F.lit(dataset_name))
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    df_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_delta_path)
    cnt = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] {bronze_delta_path:<55} (source={dataset_name}) -> {cnt:>10,} lignes")

[BRONZE OK] /lakehouse/bronze/airlines                              (source=default) ->         14 lignes


[BRONZE OK] /lakehouse/bronze/airports                              (source=default) ->        322 lignes


[BRONZE OK] /lakehouse/bronze/cancellation_codes                    (source=default) ->          4 lignes


[BRONZE OK] /lakehouse/bronze/flights                               (source=default) ->  5,819,079 lignes


In [38]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)

Tables Delta Bronze validées : ['airlines', 'airports', 'cancellation_codes', 'flights']
